# 01 Speaker verification on far-field voice-assistant audio (SLURP) + VoxCeleb1-O check

In [ ]:
# Settings: GPU T4 x2, Internet ON. Persistent outputs: results/ and runs/ in /kaggle/working.
import os, shutil, subprocess
REPO_URL = "https://github.com/ByteBoss123/speaker-aware-speech-llm.git"
WORK = "/kaggle/working/speaker-aware-speech-llm"
if not os.path.exists(WORK):
    src = "/kaggle/input/speaker-aware-speech-llm"   # alternative: upload the repo as a Kaggle dataset
    if os.path.exists(src):
        shutil.copytree(src, WORK)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
DATA = "/tmp/data"   # large ephemeral disk; /kaggle/working is capped at ~20 GB
os.environ["DATA"] = DATA

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
!bash scripts/prepare_data.sh $DATA slurp vox1test

Pipeline check: the pretrained model card reports 0.80% EER on VoxCeleb1-O (cleaned)

In [ ]:
!python scripts/sv_vox.py --vox_root $DATA/vox1 --out results/sv_vox1o.json

Speaker-disjoint split (the official SLURP splits share speakers)

In [ ]:
!python scripts/sv_slurp.py split --slurp_dir $DATA/slurp_repo/dataset/slurp --out_dir results/sv

In [ ]:
!python scripts/sv_slurp.py eval --slurp_dir $DATA/slurp_repo/dataset/slurp --audio_dir $DATA/slurp_audio/slurp_real --out_dir results/sv --tag pretrained

In [ ]:
!python scripts/sv_slurp.py noise --slurp_dir $DATA/slurp_repo/dataset/slurp --audio_dir $DATA/slurp_audio/slurp_real --out_dir results/sv --tag pretrained

DDP throughput: same job on 1 and 2 GPUs (300 steps each, not saved)

In [ ]:
!torchrun --nproc_per_node 1 scripts/sv_finetune_ddp.py --slurp_dir $DATA/slurp_repo/dataset/slurp --audio_dir $DATA/slurp_audio/slurp_real --cache_dir $DATA/slurp_cache --max_steps 300 --save 0 --tag ws1
!torchrun --nproc_per_node 2 scripts/sv_finetune_ddp.py --slurp_dir $DATA/slurp_repo/dataset/slurp --audio_dir $DATA/slurp_audio/slurp_real --cache_dir $DATA/slurp_cache --max_steps 300 --save 0 --tag ws2

Full multi-condition fine-tune on train-split speakers (2 GPUs)

In [ ]:
!torchrun --nproc_per_node 2 scripts/sv_finetune_ddp.py --slurp_dir $DATA/slurp_repo/dataset/slurp --audio_dir $DATA/slurp_audio/slurp_real --cache_dir $DATA/slurp_cache --epochs 2 --tag main

In [ ]:
!python scripts/sv_slurp.py eval --slurp_dir $DATA/slurp_repo/dataset/slurp --audio_dir $DATA/slurp_audio/slurp_real --out_dir results/sv --ckpt runs/ecapa_ft/embedding_model_main.pt --tag finetuned
!python scripts/sv_slurp.py noise --slurp_dir $DATA/slurp_repo/dataset/slurp --audio_dir $DATA/slurp_audio/slurp_real --out_dir results/sv --ckpt runs/ecapa_ft/embedding_model_main.pt --tag finetuned

Did fine-tuning hurt out-of-domain verification?

In [ ]:
!python scripts/sv_vox.py --vox_root $DATA/vox1 --ckpt runs/ecapa_ft/embedding_model_main.pt --out results/sv_vox1o_finetuned.json

In [ ]:
!python scripts/summarize.py results runs
!cd /kaggle/working && zip -qr results_sv.zip speaker-aware-speech-llm/results speaker-aware-speech-llm/runs -x '*.npz'